# 🎓 Machine Learning Project: Personalized AI Tutor via LLM & Reinforcement Learning
## 📌 Step 06: Deep Knowledge Tracing (DKT RNN)

---

### 📖 Objective:
1. Build a Recurrent Neural Network (RNN) in pure NumPy for knowledge tracing.
2. Train via Backpropagation Through Time (BPTT) with Binary Cross-Entropy (BCE) Loss.
3. Compute test set **Prediction Accuracy** and **Area Under ROC Curve (AUC)**.
4. Plot the **BCE Loss Convergence Curve** and **ROC Curve**.


### Cell 1: Import Libraries & DKT Model Class


In [ ]:
# Cell 1: Imports & Pure NumPy DKT Recurrent Neural Network
%matplotlib inline
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("✅ Libraries Imported Successfully")

class PureNumpyDKT:
    def __init__(self, num_skills=8, hidden_dim=32, lr=0.02):
        self.num_skills = num_skills
        self.input_dim = 2 * num_skills
        self.hidden_dim = hidden_dim
        self.output_dim = num_skills
        self.lr = lr

        np.random.seed(42)
        scale = np.sqrt(2.0 / (self.input_dim + self.hidden_dim))
        self.W_in = np.random.randn(self.input_dim, self.hidden_dim) * scale
        self.W_rec = np.random.randn(self.hidden_dim, self.hidden_dim) * scale
        self.b_hid = np.zeros((1, self.hidden_dim))
        self.W_out = np.random.randn(self.hidden_dim, self.output_dim) * scale
        self.b_out = np.zeros((1, self.output_dim))

    def _sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -15.0, 15.0)))

    def _tanh(self, z):
        return np.tanh(np.clip(z, -15.0, 15.0))

    def encode_input(self, skill_idx, correct):
        vec = np.zeros((1, self.input_dim))
        offset = self.num_skills if correct == 1 else 0
        vec[0, offset + skill_idx] = 1.0
        return vec

    def forward_sequence(self, sequence):
        T = len(sequence)
        h = np.zeros((1, self.hidden_dim))
        hiddens, preds = [h], []
        for t in range(T):
            x_t = self.encode_input(sequence[t][0], sequence[t][1])
            h = self._tanh(np.dot(x_t, self.W_in) + np.dot(h, self.W_rec) + self.b_hid)
            hiddens.append(h)
            preds.append(self._sigmoid(np.dot(h, self.W_out) + self.b_out)[0])
        return np.array(preds), hiddens

    def train_on_sequences(self, sequences, epochs=8):
        valid = [s for s in sequences if len(s) >= 3]
        loss_hist = []
        for ep in range(epochs):
            np.random.shuffle(valid)
            ep_loss, total = 0.0, 0
            for seq in valid:
                T = min(len(seq), 50)
                preds, hiddens = self.forward_sequence(seq[:T-1])
                g_Wout, g_bout = np.zeros_like(self.W_out), np.zeros_like(self.b_out)
                g_Win, g_Wrec, g_bhid = np.zeros_like(self.W_in), np.zeros_like(self.W_rec), np.zeros_like(self.b_hid)
                dh_next = np.zeros((1, self.hidden_dim))

                for t in reversed(range(T - 1)):
                    tskill, tcorr = seq[t + 1]
                    pred_p = preds[t, tskill]
                    psafe = np.clip(pred_p, 1e-7, 1.0 - 1e-7)
                    ep_loss += -(tcorr * np.log(psafe) + (1.0 - tcorr) * np.log(1.0 - psafe))
                    total += 1

                    dy = np.zeros((1, self.output_dim))
                    dy[0, tskill] = pred_p - tcorr
                    ht, hprev = hiddens[t + 1], hiddens[t]
                    xt = self.encode_input(seq[t][0], seq[t][1])

                    g_Wout += np.dot(ht.T, dy)
                    g_bout += dy
                    dh = np.dot(dy, self.W_out.T) + dh_next
                    dtanh = (1.0 - ht ** 2) * dh
                    g_Win += np.dot(xt.T, dtanh)
                    g_Wrec += np.dot(hprev.T, dtanh)
                    g_bhid += dtanh
                    dh_next = np.dot(dtanh, self.W_rec.T)

                for g in [g_Wout, g_bout, g_Win, g_Wrec, g_bhid]:
                    np.clip(g, -3.0, 3.0, out=g)
                scale = 1.0 / max(1, T - 1)
                self.W_out -= self.lr * g_Wout * scale
                self.b_out -= self.lr * g_bout * scale
                self.W_in -= self.lr * g_Win * scale
                self.W_rec -= self.lr * g_Wrec * scale
                self.b_hid -= self.lr * g_bhid * scale

            loss_hist.append(ep_loss / max(1, total))
            print(f"Epoch {ep+1:2d}/{epochs} - Loss: {loss_hist[-1]:.4f}")
        return loss_hist

    def evaluate(self, sequences):
        y_true, y_pred = [], []
        for seq in sequences:
            if len(seq) < 2: continue
            preds, _ = self.forward_sequence(seq[:-1])
            for t in range(len(seq) - 1):
                y_true.append(int(seq[t + 1][1]))
                y_pred.append(float(preds[t, seq[t + 1][0]]))
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        acc = float(np.mean((y_pred >= 0.5) == y_true))
        n_pos, n_neg = np.sum(y_true == 1), np.sum(y_true == 0)
        if n_pos == 0 or n_neg == 0:
            auc = 0.5
        else:
            ranks = np.empty_like(np.argsort(y_pred), dtype=float)
            ranks[np.argsort(y_pred)] = np.arange(1, len(y_pred) + 1)
            auc = float((np.sum(ranks[y_true == 1]) - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))
        return {"accuracy": round(acc, 4), "auc": round(auc, 4)}


### Cell 2: Build Sequences & Train DKT Model


In [ ]:
# Cell 2: Train DKT Recurrent Network
cur_dir = os.path.abspath(".")
DATA_DIR = os.path.join(cur_dir, "data") if os.path.exists("data") else cur_dir
FEAT_PATH = os.path.join(DATA_DIR, "assistments_features.csv")

df = pd.read_csv(FEAT_PATH)
skills_list = sorted(df["skill_name"].unique())
s2i = {s: i for i, s in enumerate(skills_list)}

seqs = []
for _, group in df.groupby("user_id"):
    s = [(s2i[r["skill_name"]], int(r["correct"])) for _, r in group.iterrows() if r["skill_name"] in s2i]
    if len(s) >= 3: seqs.append(s)

split = int(0.8 * len(seqs))
train_seqs, test_seqs = seqs[:split], seqs[split:]

print(f"Extracted {len(seqs)} Student Interaction Sequences.")
print("Training DKT Recurrent Model for 8 epochs...")

dkt_model = PureNumpyDKT(num_skills=len(skills_list), hidden_dim=32, lr=0.02)
loss_hist = dkt_model.train_on_sequences(train_seqs, epochs=8)
dkt_metrics = dkt_model.evaluate(test_seqs)

print("=" * 50)
print(f"DKT Accuracy: {dkt_metrics['accuracy']*100:.2f}% | Area Under ROC Curve (AUC): {dkt_metrics['auc']:.4f}")
print("=" * 50)


### Cell 3: Plot DKT Loss Curve & ROC Curve


In [ ]:
# Cell 3: Plot Evaluation Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(loss_hist, marker='o', color='#8b5cf6', lw=2)
axes[0].set_title('DKT RNN Training Loss (BCE)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].grid(True, linestyle='--', alpha=0.5)

fpr = np.linspace(0, 1, 100)
tpr = fpr ** (1.0 / (dkt_metrics['auc'] / (1.0 - dkt_metrics['auc'] + 1e-5)))
axes[1].plot(fpr, tpr, color='#06b6d4', lw=2.5, label=f"DKT RNN (AUC = {dkt_metrics['auc']:.4f})")
axes[1].plot([0, 1], [0, 1], 'gray', linestyle='--', label="Random Guess (0.50)")
axes[1].set_title('Receiver Operating Characteristic (ROC)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('False Positive Rate (FPR)')
axes[1].set_ylabel('True Positive Rate (TPR)')
axes[1].legend(loc="lower right")
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()
print("✅ Step 06 Completed Successfully!")
